In [1]:
import numpy as np
import random as rnd
import trax
from trax import layers as tl
from trax.supervised import training
from trax.fastmath import numpy as fastnp
import pandas as pd
import nltk
import os

rnd.seed(123)

In [2]:
# import  data
data = pd.read_csv("./questions.csv")
display(data.head())
# exploring data
print(f"# Data Pairs: {len(data)}")
print(f"# Null values: {data.isna().sum()}")

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# Data Pairs: 404351
# Null values: id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64


In [3]:
# data splitting
N_train = 300000
N_test = 1024 * 10
train_data = data[:N_train]
test_data = data[N_train:N_train + N_test]

print(f"# Train data: {len(train_data)}")
print(f"# Test data: {len(test_data)}")

print()
print("Train data")
display(train_data.head())

print()
print("Test data")
display(test_data.head())

# deleta data for memory
del(data)

# Train data: 300000
# Test data: 10240

Train data


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0



Test data


,id,qid1,qid2,question1,question2,is_duplicate
300000,300000,589215,589216,How do I prepare for interviews for cse?,What is the best way to prepare for cse?,0
300001,300001,589217,589218,What is the best bicycle to buy under 10k?,Which is the best bike in in dia to buy in INR...,1
300002,300002,589219,589220,How do I become Mutual funds distributer for a...,How do I become mutual funds distributor for a...,1
300003,300003,589221,589222,Will this relationship work?,Relationship: Will this relationship work?,0
300004,300004,589223,589224,How does Brexit affect India?,Will the GBP/AUD be affected by Brexit?,0


In [4]:
# duplicate questions
td_index = (train_data["is_duplicate"] == 1).to_numpy()
td_index = [i for i, x in enumerate(td_index) if x]
print(f"# Duplicates: {len(td_index)}")
print(f"First 10 duplicates in train_data: {td_index[:10]}")

# Duplicates: 111486
First 10 duplicates in train_data: [5, 7, 11, 12, 13, 15, 16, 18, 20, 29]


In [5]:
print(train_data["question1"][5])
print(train_data["question2"][5])
print(f"is duplicate: {train_data['is_duplicate'][5]}")

Astrology: I am a Capricorn Sun Cap moon and cap rising...what does that say about me?
I'm a triple Capricorn (Sun, Moon and ascendant in Capricorn) What does this say about me?
is duplicate: 1


In [6]:
# extracting duplicate questions batches
Q1_train_words = np.array(train_data['question1'][td_index])
Q2_train_words = np.array(train_data['question2'][td_index])

Q1_test_words = np.array(test_data['question1'])
Q2_test_words = np.array(test_data['question2'])
y_test  = np.array(test_data['is_duplicate'])

print('TRAINING QUESTIONS:\n')
print('Question 1: ', Q1_train_words[0])
print('Question 2: ', Q2_train_words[0], '\n')
print('Question 1: ', Q1_train_words[5])
print('Question 2: ', Q2_train_words[5], '\n')

print('TESTING QUESTIONS:\n')
print('Question 1: ', Q1_test_words[0])
print('Question 2: ', Q2_test_words[0], '\n')
print('is_duplicate =', y_test[0], '\n')

TRAINING QUESTIONS:

Question 1:  Astrology: I am a Capricorn Sun Cap moon and cap rising...what does that say about me?
Question 2:  I'm a triple Capricorn (Sun, Moon and ascendant in Capricorn) What does this say about me? 

Question 1:  What would a Trump presidency mean for current international master’s students on an F1 visa?
Question 2:  How will a Trump presidency affect the students presently in US or planning to study in US? 

TESTING QUESTIONS:

Question 1:  How do I prepare for interviews for cse?
Question 2:  What is the best way to prepare for cse? 

is_duplicate = 0 



In [7]:
Q1_train = np.empty_like(Q1_train_words)
Q2_train = np.empty_like(Q2_train_words)

Q1_test = np.empty_like(Q1_test_words)
Q2_test = np.empty_like(Q2_test_words)

In [8]:
# building vocabulary
from collections import defaultdict

# handle oov
vocab = defaultdict(lambda: 0)
vocab['<PAD>'] = 1

# loop through indices of train_words
for idx in range(len(Q1_train_words)):
    # tokenize words at index i in Q1_train_words and store it in Q1_train
    Q1_train[idx] = nltk.word_tokenize(Q1_train_words[idx])
    # tokenize words at index i in Q2_train_words and store it in Q2_train
    Q2_train[idx] = nltk.word_tokenize(Q2_train_words[idx])
    # concatenate tokenized words in single variable
    q = Q1_train[idx] + Q2_train[idx]
    # store word in vocab
    for word in q:
        if word not in vocab:
            # assign vocab token to each word if not in vocab
            vocab[word] = len(vocab) + 1

print(f"Length of vocab: {len(vocab)}")

Length of vocab: 36268


In [9]:
print(f"<PAD> {vocab['<PAD>']}")
print(f"Astrology {vocab['Astrology']}")
print(f"Astronomy {vocab['Astronomy']}")

<PAD> 1
Astrology 2
Astronomy 0


In [74]:
# Saving vocab for later use
import pickle

# Convert defaultdict to a regular dict
vocab_regular = dict(vocab)

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab_regular, f)

In [10]:
# loop through indices of Q1_test_words
for idx in range(len(Q1_test_words)):
    Q1_test[idx] = nltk.word_tokenize(Q1_test_words[idx])
    Q2_test[idx] = nltk.word_tokenize(Q2_test_words[idx])

In [11]:
print(f"Length of tokenized test words: {len(Q1_test)}")

Length of tokenized test words: 10240


In [12]:
# converting question into tensors
for idx in range(len(Q1_train)):
    # replace word in each sublist to a vocab index
    Q1_train[idx] = [vocab[word] for word in Q1_train[idx]]
    Q2_train[idx] = [vocab[word] for word in Q2_train[idx]]

for idx in range(len(Q1_test)):
    # replace word in each sublist to a vocab index
    Q1_test[idx] = [vocab[word] for word in Q1_test[idx]]
    Q2_test[idx] = [vocab[word] for word in Q2_test[idx]]

In [13]:
print("First question tensors in Q1_train")
print(Q1_train[0])
print("First question tensors in Q1_test")
print(Q1_test[0])

First question tensors in Q1_train
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
First question tensors in Q1_test
[32, 38, 4, 107, 65, 1015, 65, 11509, 21]


In [14]:
# splitting train dta into train and validation
cutoff = int(len(Q1_train) * 0.8)
# split data into 80% train and 20% validation
train_Q1, train_Q2 = Q1_train[:cutoff], Q2_train[:cutoff]
val_Q1, val_Q2 = Q1_train[cutoff:], Q2_train[cutoff:]

print(f"Number of duplicates: {len(Q1_train)}")
print(f"Number of duplicates in train set: {len(train_Q1)}")
print(f"Number of duplicates in validation set: {len(val_Q1)}")

Number of duplicates: 111486
Number of duplicates in train set: 89188
Number of duplicates in validation set: 22298


In [15]:
# dat6a_generator function to returnj tuple of 2 arrays with each has batch_size questions
def data_generator(Q1, Q2, batch_size, pad = 1, shuffle = True):
    # initialize inputs to empty lists
    input1 = []
    input2 = []
    
    # initialize index to 0
    index = 0
    
    # get questions indexes
    q_indexes = [*range(len(Q1))]
    
    # if shuffle is True shuffle q_indexes
    if shuffle:
        rnd.shuffle(q_indexes)

    while True:
        if index >= len(Q1):
            # set index to 0
            index = 0
            
            # shuffle data if shuffle set to True
            if shuffle:
                rnd.shuffle(q_indexes)

        # get questions from Q1, Q2 by using index from q_indexes
        q1 = Q1[q_indexes[index]]
        q2 = Q2[q_indexes[index]]
        # increment index by 1
        index += 1
        # append q1 and q2 to inputs
        input1.append(q1)
        input2.append(q2)
        
        # if inputs reached to batch_size
        if len(input1) == batch_size:
            # get max question length in both inputs then we take the max of two of them
            max_len = max(len(max(input1, key = len)), len(max(input2, key = len)))
            # ceil max_len to power of 2
            max_len = 2 ** int(np.ceil(np.log2(max_len)))
            
            # initialize b1 and b2 to empty lists for storing padded questions
            b1 = []
            b2 = []
            
            # get q1, q2 from inputs
            for q1, q2 in zip(input1, input2):
                # pad q1 until it reaches max_len
                q1 = q1 + [pad] * (max_len - len(q1))
                # pad q2 until it reaches max_len
                q2 = q2 + [pad] * (max_len - len(q2))
                # append q1 and q2 to b1 and b2
                b1.append(q1)
                b2.append(q2)
                
            # return b1 and b2 as tuple of numpy arrays
            yield (np.array(b1), np.array(b2))
            
            # reset batches
            input1, input2 = [], []

In [16]:
# test data_generator function
b_size = 2
res1, res2 = next(data_generator(train_Q1, train_Q1, b_size))

print("First question batch: ")
print(res1)

print()
print("Second question batch: ")
print(res2)

First question batch: 
[[ 541   81   62    3  491  156    6  497 2532   65   78  243   21    1
     1    1]
 [  30  156    6 1400   21    1    1    1    1    1    1    1    1    1
     1    1]]

Second question batch: 
[[ 541   81   62    3  491  156    6  497 2532   65   78  243   21    1
     1    1]
 [  30  156    6 1400   21    1    1    1    1    1    1    1    1    1
     1    1]]


In [17]:
# Siamese() model
def siamese(vocab_size = len(vocab), d_model = 128, mode = "train"):
    # normalize function (L2 norm)
    def normalize(x):
        return x / fastnp.sqrt(fastnp.sum(x * x, axis = -1, keepdims = True))
        
    # q_processor is a subnetwork used to process the question
    q_processor = tl.Serial(
        # convert input tensor to embedded tensor of shape [vocab_size, d_model]
        tl.Embedding(vocab_size, d_model),
        # fed embedded tensors to LSTM layer
        tl.LSTM(d_model),
        # take the average value across columns
        tl.Mean(axis = 1),
        # normalize thd averaged matrix
        tl.Fn('Normalize', lambda x: normalize(x))
        # returns [batch_size, d_model]
    )
    
    # model has 2 subnetworks share parameters in parallel
    model = tl.Parallel(q_processor, q_processor)
    # return model
    return model

In [18]:
# test model
model = siamese()
display(model)

Parallel_in2_out2[
  Serial[
    Embedding_41698_128
    LSTM_128
    Mean
    Normalize
  ]
  Serial[
    Embedding_41698_128
    LSTM_128
    Mean
    Normalize
  ]
]

In [85]:
# Tripletloss
def tripletlossFn(v1, v2, margin = 0.5):
    # dot product of v1 and v2.T
    scores = fastnp.dot(v1, v2.T)
    # get the batch size
    batch_size = len(scores)
    # get positive entries from scores (indicates the distance between anchor and positives)
    positive = fastnp.diagonal(scores)
    # to get the negatives without positives we subtract the scores from 2 * fastnp.eye(batch_size)
    negative_without_positive = scores - fastnp.eye(batch_size) * 2.0
    # get the closest negative score (row by row)
    closest_negative = negative_without_positive.max(axis = 1)
    negative_zero_on_duplicate = fastnp.multiply((1.0 - fastnp.eye(batch_size)), scores)
    # get the first term (mean negtive)
    mean_negative = fastnp.sum(negative_zero_on_duplicate, axis = 1) / (batch_size - 1)
    # get the first loss (max(-diff + mean_negative + margin, 0))
    tripletloss_1 = fastnp.maximum(margin - positive + mean_negative, 0)
    # get the second loss (max(margin - positive + closest_negative, 0))
    tripletloss_2 = fastnp.maximum(margin - positive + closest_negative, 0)
    # average of two losses
    avg_loss = fastnp.mean(tripletloss_1 + tripletloss_2)
    
    return avg_loss

In [86]:
v1 = np.array([[0.26726124, 0.53452248, 0.80178373],[0.5178918 , 0.57543534, 0.63297887]])
v2 = np.array([[ 0.26726124,  0.53452248,  0.80178373],[-0.5178918 , -0.57543534, -0.63297887]])
tripletlossFn(v2,v1)
print("Triplet Loss:", tripletlossFn(v2,v1))

Triplet Loss: 1.0


In [87]:
# make tripletloss layer
from functools import partial
def TripletLoss(margin = 0.25):
    tripletloss_fn = partial(tripletlossFn, margin = margin)
    return tl.Fn("TripletLoss", tripletloss_fn)

### Training

In [88]:
# get train and validation data
BATCH_SIZE = 128
train_generator = data_generator(train_Q1, train_Q1, BATCH_SIZE)
val_generator = data_generator(val_Q1, val_Q2, BATCH_SIZE)

print(f"train_Q1.shape: {train_Q1.shape}")
print(f"val_Q1.shape: {val_Q1.shape}")

train_Q1.shape: (89188,)
val_Q1.shape: (22298,)


In [90]:
# define learning rate scheduler
lr_scheduler = trax.lr.warmup_and_rsqrt_decay(400, 0.01)

def train_model(model, TripletLoss, lr_scheduler, train_generator = train_generator, val_generator = val_generator, n_steps = 1, output_dir = "model/"):
    # define train_task
    train_task = training.TrainTask(
        labeled_data = train_generator,
        loss_layer = TripletLoss(),
        optimizer = trax.optimizers.Adam(0.01),
        lr_schedule = lr_scheduler,
        n_steps_per_checkpoint = 50
    )
    # define eval_task
    # eval_task = training.EvalTask(
    #     labeled_data = val_generator,
    #     metrics = [TripletLoss()]
    # )
    eval_task = training.EvalTask(
        labeled_data=val_generator,
        metrics=[TripletLoss()],
    )
    # define training loop
    training_loop = training.Loop(
        model,
        train_task,
        eval_tasks=[eval_task],
        output_dir = output_dir
    )
    # run training loop for # steps
    training_loop.run(n_steps)
    return training_loop

In [ ]:
# train model
siamese_model = siamese()
training_loop = train_model(siamese_model, TripletLoss, lr_scheduler, n_steps = 500, output_dir = "model_5/")


Step      1: Total number of trainable weights: 5468928
Step      1: Ran 1 train steps in 68.29 secs
Step      1: train TripletLoss |  0.49997032
Step      1: eval  TripletLoss |  0.49999499

Step     50: Ran 49 train steps in 275.64 secs
Step     50: train TripletLoss |  0.49967688
Step     50: eval  TripletLoss |  0.49910104


### Evaluation

In [60]:
# Load trained model
def load_trained_siamese(model_path):
    model = siamese()
    model.init_from_file(model_path, weights_only=True)
    return model

In [68]:
s_model = load_trained_siamese("model_3/model.pkl.gz")

In [69]:
def classify(test_Q1, test_Q2, y, threshold, model, vocab, data_generator=data_generator, batch_size = 64):
    # initialize accuracy with 0 at start
    accuracy = 0
    # loop through each Q1 eacdh batch_size step
    for i in range(0, len(test_Q1), batch_size):
        q1, q2 = next(data_generator(test_Q1[i:i+batch_size], test_Q2[i:i+batch_size], batch_size, pad = vocab["<PAD>"], shuffle = False))
        # chunk of y_test
        y_test = y[i:i+batch_size]
        # get v1 and v2
        v1, v2 = model([q1, q2])
        # loop through batch_size range
        for j in range(batch_size):
            # compute cosine similarity between elements of 2 vectors
            d = np.dot(v1[j], v2[j].T)
            # check if d > threshold
            res = d > threshold
            # increment accuracy if y_test[j] == res
            accuracy += (y_test[j] == res)

    # get average of accuracy
    accuracy = accuracy / len(test_Q1)
    
    # return accuracy
    return accuracy            

In [77]:
accuracy = classify(Q1_test, Q2_test, y_test, 0.5, s_model, vocab, batch_size = 512)
print(f"Acuuracy: {accuracy}")

Acuuracy: 0.616015625


### Refrence

In [81]:
def predict(q1, q2, model, vocab = vocab, data_generator = data_generator, threshold = 0.7, verbose = False):
    # tokenize questions
    q1_tokens = nltk.word_tokenize(q1)
    q2_tokens = nltk.word_tokenize(q2)
    # create encoded words
    Q1, Q2 = [], []
    # encode q1
    for word in q1_tokens:
        # increment by checking the 'word' index in `vocab`
        Q1 += [vocab[word]]
    # encode q2
    for word in q2_tokens:
        # increment by checking the 'word' index in `vocab`
        Q2 += [vocab[word]]

    # padd Q1 and Q2
    Q1, Q2 = next(data_generator([Q1], [Q2], batch_size = 1, pad = vocab["<PAD>"], shuffle =False))
    # Get v1 and v2 from model
    v1, v2 = model([Q1, Q2])
    # compute cosine similarity
    d = np.dot(v1, v2.T)
    # get res whether they are similar or not
    res = d > threshold
    # print out the results
    if verbose:
        print(f"Question 1 = {q1}\nQuestion 2 = {q2}")
        print(f"Q1 = {Q1}\nQ2 = {Q2}")
        print(f"Similarity score = {d}")
        print(f"Are similar ? : {res}")
    return res

In [82]:
question1 = "When will I see you?"
question2 = "When can I see you again?"
res = predict(question1, question2, s_model, threshold = 0.7, verbose = True)

Question 1 = When will I see you?
Question 2 = When can I see you again?
Q1 = [[585  76   4  46  53  21   1   1]]
Q2 = [[ 585   33    4   46   53 7280   21    1]]
Similarity score = [[0.8718137]]
Are similar ? : [[ True]]


In [83]:
question1 = "Do they enjoy eating the dessert?"
question2 = "Do they like hiking in the desert?"
res = predict(question1, question2, s_model, threshold = 0.7, verbose = True)

Question 1 = Do they enjoy eating the dessert?
Question 2 = Do they like hiking in the desert?
Q1 = [[  443  1145  3159  1169    78 29017    21     1]]
Q2 = [[  443  1145    60 15302    28    78  7431    21]]
Similarity score = [[0.58603406]]
Are similar ? : [[False]]


In [84]:
question1 = "Did you enjoy the movie?"
question2 = "Was the movie good?"
res = predict(question1, question2, s_model, threshold = 0.7, verbose = True)

Question 1 = Did you enjoy the movie?
Question 2 = Was the movie good?
Q1 = [[ 478   53 3159   78  293   21    1    1]]
Q2 = [[1247   78  293   35   21    1    1    1]]
Similarity score = [[0.12992327]]
Are similar ? : [[False]]
